# 🤖 第5章：多 Agent 系统

## 🎯 学习目标
1. 理解 AgentTool 的完整架构
2. 理解内置 Agent 类型及其设计
3. 理解 Agent 的生命周期管理 (runAgent)
4. 理解同步/异步 Agent 的区别
5. 理解 Agent 间通信和团队协作

---

## 5.1 AgentTool 概览

AgentTool 是 Claude Code 中最复杂的工具之一（~1400行）。
它允许 Claude 创建**子 Agent** 来处理子任务。

### 核心概念

```
主 Agent (Main Thread)
  │
  ├── AgentTool.call({ prompt: '搜索所有 TODO', subagent_type: 'Explore' })
  │   └── 子 Agent: Explore
  │       ├── 有自己的消息历史
  │       ├── 有自己的工具集（受限）
  │       ├── 有自己的系统提示词
  │       └── 执行完毕后返回结果给主 Agent
  │
  ├── AgentTool.call({ prompt: '制定重构计划', subagent_type: 'Plan' })
  │   └── 子 Agent: Plan
  │
  └── AgentTool.call({ prompt: '运行测试验证', subagent_type: 'Verification' })
      └── 子 Agent: Verification
```

### 输入 Schema

```typescript
const inputSchema = z.object({
  description: z.string(),     // 3-5 词的任务描述
  prompt: z.string(),          // 详细的任务指令
  subagent_type: z.string().optional(), // Agent 类型
  model: z.enum(['sonnet', 'opus', 'haiku']).optional(), // 模型覆盖
  run_in_background: z.boolean().optional(), // 后台运行
  name: z.string().optional(),  // Agent 名称（团队模式）
  team_name: z.string().optional(), // 团队名称
  isolation: z.enum(['worktree', 'remote']).optional(), // 隔离模式
  cwd: z.string().optional(),   // 工作目录覆盖
});
```

## 5.2 内置 Agent 类型

Claude Code 内置了多种专用 Agent：

| Agent 类型 | 用途 | 工具集 | 只读 |
|-----------|------|--------|------|
| **Explore** | 代码搜索和探索 | Glob, Grep, Read, Bash(只读) | ✅ |
| **Plan** | 制定执行计划 | Glob, Grep, Read, Bash(只读) | ✅ |
| **Verification** | 验证代码变更 | Glob, Grep, Read, Bash | ❌ |
| **GeneralPurpose** | 通用任务 | 所有工具 | ❌ |
| **Fork** | 分叉执行 | 继承父级工具 | ❌ |

### Explore Agent 示例

```typescript
// built-in/exploreAgent.ts (简化)
export const EXPLORE_AGENT: AgentDefinition = {
  agentType: 'Explore',
  description: 'Search and explore code',
  
  // 只读工具集
  allowedTools: ['Glob', 'Grep', 'Read', 'Bash'],
  
  // 权限模式：只读
  permissionMode: 'plan',
  
  // 省略 CLAUDE.md（节省 token）
  omitClaudeMd: true,
  
  // 系统提示词
  getSystemPrompt() {
    return `You are a code exploration agent. Your job is to search 
    through the codebase and find relevant information. You should 
    use Glob, Grep, and Read tools to explore. Do NOT make any 
    changes to the code.`;
  },
};
```

**设计亮点：**
- `omitClaudeMd: true` — Explore Agent 不需要项目规则，节省 5-15 Gtok/周
- `permissionMode: 'plan'` — 强制只读，即使父级是 bypass 模式

## 5.3 Agent 生命周期 (runAgent.ts)

`runAgent.ts` (974行) 管理子 Agent 的完整生命周期：

```
AgentTool.call()
  │
  ▼
1. 选择 Agent 定义
  │  ├── 查找 subagent_type 对应的 AgentDefinition
  │  ├── 检查权限（是否被 deny）
  │  └── 检查 MCP 依赖是否满足
  │
  ▼
2. 初始化 Agent 环境
  │  ├── 创建 agentId (UUID)
  │  ├── 解析模型 (getAgentModel)
  │  ├── 构建系统提示词
  │  ├── 初始化 Agent MCP 服务器
  │  ├── 解析工具集 (resolveAgentTools)
  │  ├── 创建子上下文 (createSubagentContext)
  │  └── 预加载技能 (skills)
  │
  ▼
3. 执行 SubagentStart Hooks
  │
  ▼
4. 进入查询循环
  │  for await (const message of query({ ... })) {
  │    // 记录 sidechain transcript
  │    // 转发消息给父级
  │  }
  │
  ▼
5. 清理
  │  ├── 清理 MCP 服务器
  │  ├── 清理 session hooks
  │  ├── 清理 prompt cache tracking
  │  ├── 释放文件缓存
  │  ├── 释放 Perfetto 追踪
  │  ├── 清理 todos
  │  └── 终止后台 bash 任务
  │
  ▼
6. 返回结果给 AgentTool
```

## 5.4 同步 vs 异步 Agent

### 同步 Agent（默认）

```
主 Agent ──────────┐
                    │ 等待
子 Agent ───执行───┘ 返回结果
主 Agent ──────────── 继续
```

- 共享父级的 `setAppState`
- 共享父级的 `abortController`（父级中断 → 子级也中断）
- 可以显示权限确认对话框

### 异步 Agent（run_in_background: true）

```
主 Agent ──────────────────────── 继续工作
子 Agent ───执行──────────────── 完成后通知
```

- 独立的 `abortController`
- `setAppState` 是 no-op（不影响主线程状态）
- `shouldAvoidPermissionPrompts: true`（不能弹对话框）
- 通过 `setAppStateForTasks` 注册/更新任务状态

### 关键代码

```typescript
// runAgent.ts 中的上下文创建
const agentToolUseContext = createSubagentContext(toolUseContext, {
  agentId,
  agentType: agentDefinition.agentType,
  messages: initialMessages,
  abortController: isAsync 
    ? new AbortController()           // 异步：独立控制器
    : toolUseContext.abortController,  // 同步：共享控制器
  shareSetAppState: !isAsync,          // 异步：不共享状态
});
```

## 5.5 Agent 工具集解析

每个 Agent 类型有自己的工具集限制：

```typescript
// agentToolUtils.ts
function resolveAgentTools(agentDef, availableTools, isAsync) {
  // 1. 从可用工具中过滤
  let tools = availableTools;
  
  // 2. 移除 Agent 不允许使用的工具
  tools = tools.filter(t => !ALL_AGENT_DISALLOWED_TOOLS.has(t.name));
  // 不允许的工具：AskUserQuestion, EnterPlanMode, ExitPlanMode 等
  
  // 3. 如果 Agent 定义了 allowedTools，只保留这些
  if (agentDef.allowedTools) {
    tools = tools.filter(t => agentDef.allowedTools.includes(t.name));
  }
  
  // 4. 异步 Agent 的额外限制
  if (isAsync) {
    tools = tools.filter(t => ASYNC_AGENT_ALLOWED_TOOLS.has(t.name));
  }
  
  return { resolvedTools: tools };
}
```

### 工具限制常量

```typescript
// constants/tools.ts
export const ALL_AGENT_DISALLOWED_TOOLS = new Set([
  'AskUserQuestion',  // 子 Agent 不能直接问用户
  'EnterPlanMode',    // 子 Agent 不能切换模式
  'ExitPlanMode',     // 子 Agent 不能切换模式
  'Brief',            // 子 Agent 不能切换简要模式
]);
```

## 5.6 Agent 系统提示词构建

```typescript
async function getAgentSystemPrompt(agentDef, context, model, dirs, tools) {
  // 1. 获取 Agent 自定义提示词
  const agentPrompt = agentDef.getSystemPrompt({ toolUseContext });
  
  // 2. 增强环境信息
  return enhanceSystemPromptWithEnvDetails(
    [agentPrompt],
    model,                        // 模型名称
    additionalWorkingDirectories, // 额外工作目录
    enabledToolNames,             // 可用工具名
  );
  // enhanceSystemPromptWithEnvDetails 会添加：
  // - 操作系统信息
  // - 当前工作目录
  // - 可用工具列表
  // - 模型特定的指令
}
```

### 上下文优化

```typescript
// Explore/Plan Agent 省略 CLAUDE.md（节省 token）
const shouldOmitClaudeMd = agentDef.omitClaudeMd 
  && !override?.userContext;

// Explore/Plan Agent 省略 git status（它们可以自己运行 git）
const resolvedSystemContext = 
  (agentDef.agentType === 'Explore' || agentDef.agentType === 'Plan')
    ? systemContextNoGit 
    : baseSystemContext;
```

## 5.7 Agent MCP 服务器

Agent 可以定义自己的 MCP 服务器，与父级的服务器合并：

```typescript
// Agent 定义中的 MCP 服务器
const agentDef: AgentDefinition = {
  agentType: 'MyAgent',
  mcpServers: [
    'existing-server',           // 引用已有服务器（共享连接）
    { 'my-server': {             // 内联定义（Agent 专属）
      command: 'node',
      args: ['server.js'],
    }},
  ],
};

// initializeAgentMcpServers() 的逻辑：
// 1. 引用型：复用父级的 memoized 连接
// 2. 内联型：创建新连接，Agent 结束时清理
// 3. 合并：[...parentClients, ...agentClients]
```

## 5.8 Agent 团队 (Swarm Mode)

当 `isAgentSwarmsEnabled()` 为 true 时，Agent 可以组成团队：

```
Team: "feature-team"
  │
  ├── Leader Agent (主 Agent)
  │   └── 协调团队工作
  │
  ├── Teammate: "frontend" (tmux pane)
  │   └── 处理前端任务
  │
  └── Teammate: "backend" (tmux pane)
      └── 处理后端任务
```

### 团队通信

```typescript
// SendMessageTool — Agent 间通信
AgentTool.call({
  name: 'frontend',
  team_name: 'feature-team',
  prompt: '实现登录页面的 UI',
});

// 后续可以通过 SendMessage 发送消息
SendMessageTool.call({
  to: 'frontend',
  message: '请使用 React Hook Form 处理表单',
});
```

### 隔离模式

```typescript
// worktree 隔离：创建 git worktree
AgentTool.call({
  prompt: '重构认证模块',
  isolation: 'worktree',  // 在独立的 git worktree 中工作
});
// Agent 在独立的代码副本中工作，不影响主分支

// remote 隔离：远程环境（ant-only）
AgentTool.call({
  prompt: '运行完整测试套件',
  isolation: 'remote',    // 在远程 CCR 环境中运行
});
```

## 5.9 Coordinator Mode

当 `COORDINATOR_MODE` feature flag 启用时，系统进入协调器模式：

```
Coordinator (协调器)
  │  只有 AgentTool + TaskStopTool + SendMessageTool
  │  不直接执行任何文件/shell 操作
  │
  ├── Worker Agent 1
  │   └── 有完整工具集，执行实际工作
  │
  ├── Worker Agent 2
  │   └── 有完整工具集，执行实际工作
  │
  └── Worker Agent 3
      └── 有完整工具集，执行实际工作
```

**协调器的职责：**
- 分解任务
- 分配给 Worker
- 收集结果
- 综合输出

**Worker 的特点：**
- 有完整的工具集
- 权限由协调器的 permission handler 管理
- 可以并行执行

## 5.10 Agent 清理机制

Agent 结束时的清理非常重要，防止资源泄露：

```typescript
// runAgent.ts finally 块
try {
  for await (const message of query({ ... })) {
    yield message;
  }
} finally {
  // 1. 清理 Agent 专属 MCP 服务器
  await mcpCleanup();
  
  // 2. 清理 session hooks
  clearSessionHooks(rootSetAppState, agentId);
  
  // 3. 清理 prompt cache tracking
  cleanupAgentTracking(agentId);
  
  // 4. 释放文件缓存内存
  agentToolUseContext.readFileState.clear();
  
  // 5. 释放消息数组内存
  initialMessages.length = 0;
  
  // 6. 清理 Perfetto 追踪
  unregisterPerfettoAgent(agentId);
  
  // 7. 清理 todos
  rootSetAppState(prev => {
    const { [agentId]: _removed, ...todos } = prev.todos;
    return { ...prev, todos };
  });
  
  // 8. 终止后台 bash 任务
  killShellTasksForAgent(agentId, ...);
}
```

**为什么 `initialMessages.length = 0`？**
这是一个内存优化技巧。直接设置 length 为 0 比 `initialMessages = []` 更高效，
因为它释放了数组中所有元素的引用，让 GC 可以回收。

## ✅ 本章小结

| 概念 | 关键点 |
|------|--------|
| AgentTool | 最复杂的工具，~1400行，支持多种 Agent 类型 |
| 内置 Agent | Explore(只读搜索), Plan(规划), Verification(验证), GeneralPurpose(通用) |
| 生命周期 | 选择 → 初始化 → Hook → 查询循环 → 清理 |
| 同步/异步 | 同步共享状态和中断，异步完全隔离 |
| 工具集 | 每种 Agent 有不同的工具限制 |
| 团队模式 | tmux pane 中的多 Agent 协作 |
| Coordinator | 只协调不执行，Worker 做实际工作 |
| 清理 | 8 步清理流程，防止资源泄露 |

### 下一章预告
第6章将深入 **MCP 协议与扩展系统**——理解 Claude Code 如何通过 MCP 扩展能力。